# Step 1: Raw Data

A data lake accepts data in its original form: whatever format the source system produces, without upfront modeling. This is the inverse of a data warehouse, where data is transformed and validated before it is loaded (schema-on-write).

This notebook generates a synthetic sales export — `date, region, product, quantity, revenue` — as a single CSV file. CSV is used deliberately as the starting point: it is row-based, uncompressed, and human-readable, representative of the format typically produced by a point-of-sale or ERP export when it first lands in a lake's raw zone.

In [1]:
import os
from pathlib import Path

# Make sure we run from the repository root, regardless of the
# notebook's working directory
while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

Working directory: /workspaces/python_data_lake


## Generating the Raw CSV File

`scripts/generate_data.py` writes 300,000 rows in a series of chunks, producing a file of approximately 10–15 MB. This size is small enough to generate and query within seconds, and small enough to inspect manually in a spreadsheet, yet large enough to make differences in file format and query performance clearly measurable.

In [2]:
!python scripts/generate_data.py --out lake/raw/sales.csv

     100,000 / 300,000 rows written
     200,000 / 300,000 rows written
     300,000 / 300,000 rows written

Done in 1.4s
File:  lake/raw/sales.csv
Size:  12.4 MB
Rows:  300,000


## Inspecting the Raw Data

Locate `lake/raw/sales.csv` in the file browser and note its size. This value will be compared against the Parquet format in the next notebook.

In [3]:
import pandas as pd

csv_path = Path("lake/raw/sales.csv")
size_mb = csv_path.stat().st_size / (1024 * 1024)
print(f"{csv_path} is {size_mb:.1f} MB")

pd.read_csv(csv_path, nrows=10)

lake/raw/sales.csv is 12.4 MB


,date,region,product,quantity,revenue
0,2024-03-27,Lucerne,Power Bank,14,16015.28
1,2026-01-18,Geneva,Webcam HD,17,19518.12
2,2025-09-24,Zurich,Network Switch,11,6928.78
3,2025-02-28,Lucerne,Router,8,2616.19
4,2025-02-22,Winterthur,Office Chair,1,885.63
5,2026-04-10,Bern,Tablet 10in,12,9505.07
6,2024-03-24,Basel,Tablet 10in,18,18961.80
7,2025-11-05,Geneva,Power Bank,3,3294.24
8,2024-07-13,St. Gallen,Standing Desk,5,1791.94
9,2024-04-01,Lausanne,27in Monitor,11,7775.33


## Limitations of CSV as a Permanent Storage Format

- **Row-based storage**: reading only the `revenue` column still requires every engine to scan every byte of every row.
- **No compression**: repeated values such as region and product names are stored in full on every occurrence.
- **No embedded schema**: data types (`quantity` as an integer, `revenue` as a float) must be inferred on every read.
- **No partitioning**: a query restricted to a single month still requires reading the entire file.

The next two notebooks address these limitations — first by introducing a more efficient file format (Parquet), then by introducing a query engine (DuckDB) capable of exploiting it.